In [1]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\Mohamed Roomy\AppData\Local\Temp\ipykernel_2520\3531169212.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("data/attention.pdf")
documents = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

In [9]:
import os
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [12]:
embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Embeddings working --OK---")

Embeddings working --OK---


In [13]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/sample.pdf")
documents = loader.load()

print("Pages:", len(documents))

Pages: 15


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print("Chunks:", len(docs))

Chunks: 52


In [15]:
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)


In [18]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("Chroma DB Ready OK")

Chroma DB Ready OK


In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = ChatPromptTemplate.from_template("""
Answer based on context:

{context}

Question: {question}
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [25]:
response = rag_chain.invoke("Who are the authors of attention is all you need?")

print(response)

The authors of "Attention Is All You Need" are:

1. Ashish Vaswani (Google Brain)
2. Noam Shazeer (Google Brain)
3. Niki Parmar (Google Research)
4. Jakob Uszkoreit (Google Research)
5. Llion Jones (Google Research)
6. Aidan N. Gomez (University of Toronto)
7. Łukasz Kaiser (Google Brain)
8. Illia Polosukhin (independent researcher)


In [27]:
query = "Who are the authors of Attention Is All You Need?"

# Retrieve the top 3 similar chunks
retrieved_results = vectorstore.similarity_search(query, k=3)

print("===== Retrieved Chunks =====")

for i, doc in enumerate(retrieved_results, start=1):
    print(f"\nChunk {i}")
    print(doc.page_content)

print("\n" + "=" * 60)

# Get the final answer from the RAG chain
response = rag_chain.invoke(query)

print("===== AI Answer =====")
print(response)

===== Retrieved Chunks =====

Chunk 1
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with rec